## **RAG and Wildfire using Streamlit**

*Wildfire Generative AI*

* **RAG NLP Solution for General Wildfire Inquiries**
* **Predicition of wildifire in a specific area:**
    * Call the weather api alerts endpoint to get the active fire alerts.
    * Check NASA FIRMS Thermal Anomaly.
    * Call the custom classification model and get the prediction.


Sources:
https://www.analyticsvidhya.com/blog/2024/04/rag-and-streamlit-chatbot-chat-with-documents-using-llm/


https://medium.com/snowflake/langchain-and-streamlit-rag-c5f53af8f6ba


https://colab.research.google.com/github/sunnysavita10/Generative-AI-Indepth-Basic-to-Advance/blob/main/RAG_with_LLAMA3_1.ipynb#scrollTo=3Y_UIpLVzBrI

https://www.weather.gov/documentation/services-web-api


In [ ]:
!pip uninstall -y langchain langchain-core langchain-community langchain-huggingface requests
!pip install --upgrade langchain
!pip install langchain-classic
!pip install langchain-huggingface
!pip install langchain-community
!pip install langchain-core
!pip install transformers
!pip install torch
!pip install us
!pip install huggingface
!pip install huggingface_hub
!pip install sentence-transformers
!pip install faiss-gpu-cu11
!pip install pypdf
!pip install requests
!pip install unstructured
!pip install streamlit

Found existing installation: langchain 0.3.27
Uninstalling langchain-0.3.27:
  Successfully uninstalled langchain-0.3.27
Found existing installation: langchain-core 0.3.79
Uninstalling langchain-core-0.3.79:
  Successfully uninstalled langchain-core-0.3.79
Found existing installation: requests 2.32.4
Uninstalling requests-2.32.4:
  Successfully uninstalled requests-2.32.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 467.1/467.1 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.4/155.4 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.6/207.6 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into accoun

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.9/323.9 kB 9.9 MB/s eta 0:00:00


In [1]:
#Get the Huggingface Token and set the environment variable
from transformers import pipeline
from google.colab import userdata
import requests
import os
HUGGINGFACEHUB_API_TOKEN = userdata.get("token1studio")
os.environ["HUGGINGFACEHUB_API_TOKEN"] = HUGGINGFACEHUB_API_TOKEN
NASA_FIRMS_KEY = userdata.get("NASAfirms")
os.environ["NASA_FIRMS_KEY"] = NASA_FIRMS_KEY

print(HUGGINGFACEHUB_API_TOKEN)
print(NASA_FIRMS_KEY)

NotebookAccessError: Notebook does not have access to secret token1studio

In [ ]:
%%writefile user_intent.py

#zero shot classification
from transformers import pipeline

def user_intent(prompt):

  #Load a zero shot pipeline
  classifier = pipeline("zero-shot-classification",
                        model="facebook/bart-large-mnli")

  #Define the intents
  candidate_labels = [
      "area-specific wildfire risk prediction",   # predicting fire risk at a location
      "wildfire safety and prevention guidance",  # how to prepare, prevent, stay safe
      "wildfire general information",             # background, causes, definitions
      "wildfire preparedness steps"               # concrete actions to take before/during
  ]

  #Classify an example query
  #query = "Will there be a wildfire near Los Angeles next month?"
  result = classifier(prompt, candidate_labels)

  #Pick the highest-scoring label
  best = result["labels"][0]
  return best


Writing user_intent.py


This pipeline lets users ask general wildfire questions and get precise, context-grounded answers drawn from your loaded documents

In [ ]:
%%writefile RAG_LLM.py
import os
import re
from langchain_community.document_loaders import PyPDFLoader, CSVLoader, UnstructuredURLLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_community.memory import ConversationBufferWindowMemory
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.llms import HuggingFacePipeline
from langchain_community.llms import HuggingFaceEndpoint
from transformers import pipeline

# Configuration
EMBEDDING_MODEL = "sentence-transformers/all-mpnet-base-v2"
EMBEDDING_KWARGS = {"device": "cuda"}
LLM_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"
PDF_PATHS = [
    "/content/sample_data/facnet_improvefireoutcomes.pdf",
    "/content/sample_data/Firewise_Band_Together_Toolkit.pdf",
    "/content/sample_data/wildfire-evacuation-checklist.pdf"
]
csv_paths = ["/content/sample_data/wildfire_resources_for_state.csv"]
URLS = [
    "https://www.nj.gov/dep/parksandforests/fire/program/aboutrxb.html",
    "https://fire.sref.info/",
    "https://www.usfa.fema.gov/wui/outreach/wildfire-evacuation.html",
    "https://www.usfa.fema.gov/wui/"
]

# Define custom RAG prompt
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a wildfire assistant that will provide wildfire safety tips and general wildfire inquiries, covering risk awareness, prevention, and preparedness. Answer using **only** the retrieved context. Do **NOT** invent facts. If the answer cannot be found in the context, respond: 'I’m not sure based on the provided information.'"),
    ("user", "Given the context: {context}\nAnswer the following question: {question}\n###BEGIN_ANSWER###")
])

# Document Ingestion & Chunking
all_documents = []
for path in PDF_PATHS:
    if os.path.exists(path):
        loader = PyPDFLoader(path)
        documents = loader.load()
        if documents:
            all_documents.extend(documents)
for path in csv_paths:
    if os.path.exists(path):
        loader = CSVLoader(path)
        documents = loader.load()
        if documents:
            all_documents.extend(documents)
for url in URLS:
    loader = UnstructuredURLLoader([url])
    documents = loader.load()
    if documents:
        all_documents.extend(documents)

# Text Chunking
splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=30)
docs = splitter.split_documents(all_documents)

# Embedding & FAISS Indexing
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL, model_kwargs=EMBEDDING_KWARGS)
vector_store = FAISS.from_documents(docs, embeddings)
vector_store.save_local("vector_store/multi_docs")
vector_store = FAISS.load_local("vector_store/multi_docs", embeddings, allow_dangerous_deserialization=True)


HF_TOKEN = os.environ["HUGGINGFACEHUB_API_TOKEN"]

# llm = HuggingFaceHub(
#     repo_id=LLM_MODEL,
#     model_kwargs={
#         "temperature": 1,
#         "max_length": 400,
#         "request_timeout": 300,
#         "stop": ["###END_ANSWER###"]
#     },
#     huggingfacehub_api_token=HUGGINGFACEHUB_API_TOKEN
# )


# Load HF model using pipeline and wrap with HuggingFacePipeline
#generator = pipeline(
    #"text-generation",
    #model=LLM_MODEL,
    #token=HF_TOKEN,
    #device=-1,  # <== force CPU
    #max_new_tokens=400,
    #temperature=1,
    #do_sample=True
#)
#llm = HuggingFacePipeline(pipeline=generator)

# Call HuggingFaceEndpoint
llm = HuggingFaceEndpoint(
    repo_id="mistralai/Mistral-7B-Instruct-v0.3",
    task="text-generation",
    huggingfacehub_api_token=os.environ["HUGGINGFACEHUB_API_TOKEN"],
    temperature=1,
    max_new_tokens=400,
    repetition_penalty=1.03,
    stop_sequences=["###END_ANSWER###"]
)

# Memory
memory = ConversationBufferWindowMemory(k=2, memory_key="chat_history", output_key="answer", return_messages=True)

# Conversational RAG Chain
qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=vector_store.as_retriever(),
    memory=memory,
    chain_type="stuff",
    condense_question_prompt=ChatPromptTemplate.from_messages([
        ("system", "You generate follow‑ups into standalone questions."),
        ("user", "History:\n{chat_history}\nFollow‑up: {question}")
    ]),
    combine_docs_chain_kwargs={"prompt": rag_prompt},
    return_source_documents=True,
)

# Query Function
def qa_conversation(question: str) -> str:
    result = qa_chain({"question": question}, return_only_outputs=True)
    raw = result["answer"]
    pattern = r"###BEGIN_ANSWER###(.*?)###END_ANSWER###"
    matches = re.findall(pattern, raw, re.DOTALL)
    if matches:
       return matches[-1].strip()
    else:
       return raw.strip()


Overwriting RAG_LLM.py


From a place name figure out the latitude, longitude, and address using the geopy.geocoders Nominatim library.

In [ ]:
%%writefile get_coordinates.py
from geopy.geocoders import Nominatim

def get_coordinates(location):
  """
  Returns the latitude and longitude of a given location.

  Args:
    location: A string representing the location (e.g., "Santa Fe, NM").

  Returns:
    A tuple containing the latitude and longitude if the location is found,
    otherwise None.
  """
  geolocator = Nominatim(user_agent="my_geolocator") # You can replace "my_geolocator" with a more descriptive name
  try:
    location_info = geolocator.geocode(location)
    if location_info:

      return round(location_info.latitude, 4), round(location_info.longitude, 4), location_info.address
    else:
      return None
  except Exception as e:
    print(f"Error: {e}")
    return None

# Example usage:
place_name = "Santa Fe, NM"
coordinates = get_coordinates(place_name)

if coordinates:
  latitude, longitude, address = coordinates
  print(f"The latitude and longitude for {place_name} are: {latitude}, {longitude}")
else:
  print(f"Could not find coordinates for {place_name}")

Writing get_coordinates.py


Use a zero-shot classification to figure out the state name from the address.

In [ ]:
%%writefile get_state.py
from transformers import pipeline
def get_state(address):
# Zero‑shot pipeline (MNLI model)
  classifier = pipeline(
      "zero-shot-classification",
      model="facebook/bart-large-mnli"      # or any other MNLI‑style model
  )

  # All 50 U.S. states as labels
  US_STATES = [
      "Alabama", "Alaska", "Arizona", "Arkansas", "California", "Colorado",
      "Connecticut", "Delaware", "Florida", "Georgia", "Hawaii", "Idaho",
      "Illinois", "Indiana", "Iowa", "Kansas", "Kentucky", "Louisiana",
      "Maine", "Maryland", "Massachusetts", "Michigan", "Minnesota",
      "Mississippi", "Missouri", "Montana", "Nebraska", "Nevada",
      "New Hampshire", "New Jersey", "New Mexico", "New York",
      "North Carolina", "North Dakota", "Ohio", "Oklahoma", "Oregon",
      "Pennsylvania", "Rhode Island", "South Carolina", "South Dakota",
      "Tennessee", "Texas", "Utah", "Vermont", "Virginia", "Washington",
      "West Virginia", "Wisconsin", "Wyoming"
  ]

  # Your text
  #text = "Marlton, Evesham Township, Burlington County, New Jersey, United States"

  # Run zero‑shot classification
  res = classifier(
      address,
      candidate_labels=US_STATES,
      multi_class=False     # exactly one label
  )

  # Pick the top label
  state = res["labels"][0]
  score = res["scores"][0]

  #print(f"Detected state: {state}  (score: {score:.3f})")
  return state


Writing get_state.py


 When given a string, this code finds every place name in it, and hands back a neat, comma‑delimited string of those locations.

In [ ]:
%%writefile city_country_ner.py

from transformers import pipeline
def ner(prompt):
  generator = pipeline(
      task="ner",
      model="cjber/reddit-ner-place_names",
      tokenizer="cjber/reddit-ner-place_names",
      aggregation_strategy="first",
  )

  ner_results = generator(prompt)

  #Extract location words into a list
  locations = []

  for item in ner_results:
    if item['entity_group'] == 'location':
      locations.append(item['word'])

  #Build a single string of all locations separated by commas (with a trailing comma)
  location_string = ""

  for loc in locations:
    location_string += loc + ", "

  location_string = location_string[:-2] + "."

  #print(location_string)
  return location_string

  #print(out)


Writing city_country_ner.py


Look up the two‑letter postal abbreviation for a U.S. state name.
Returns None if the state isn't found.

In [ ]:
%%writefile state_abbreviation.py

import us

def get_state_abbr(name: str) -> str | None:

    st = us.states.lookup(name)
    return st.abbr if st else None

# Examples
#print(get_state_abbr("New Jersey"))   # → "NJ"
#print(get_state_abbr("California"))   # → "CA"
#print(get_state_abbr("new jersey"))   # → "NJ"  (case‑insensitive)


Writing state_abbreviation.py


In [ ]:
%%writefile weather_api_alerts.py
import json

def weather_api_alerts(weather):

  # Iterate through each feature and pick out fire warnings
  fire_warnings = []

  for feature in weather.get('features', []):
      props = feature.get('properties', {})
      event = props.get('event', '')
      description = props.get('description', '')
      # Check for the word fire in description
      if 'fire' in description.lower():
          fire_warnings.append(props)
      # Filter for Red Flag Warning (fire‑weather)
      #if event == "Red Flag Warning" or event == "Fire Weather Watch" or "Special Weather Statement":
          #fire_warnings.append(props)

  # Check if fire_warnings has any value or is empty
  if not fire_warnings:
      fire_warnings_str = "There are no fire alerts in your state today."
      return fire_warnings_str
  else:
      return fire_warnings


Writing weather_api_alerts.py


Calculates the distance between two points on Earth given their latitude and longitude coordinates. Returns the distance between the two points in miles.

In [ ]:
%%writefile haversine_dist.py
import math

def distance(lat1, lon1, lat2, lon2):

  # Radius of the Earth in miles
  R = 3958.76

  # Convert latitude and longitude from degrees to radians
  lat1_rad = math.radians(lat1)
  lon1_rad = math.radians(lon1)
  lat2_rad = math.radians(lat2)
  lon2_rad = math.radians(lon2)

  # Haversine formula
  a = math.sin((lat2_rad - lat1_rad) / 2)**2 + math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin((lon2_rad - lon1_rad) / 2)**2
  c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
  distance = R * c

  return distance

Writing haversine_dist.py


Source of the thermal anomaly: NASA FIRMS Classification.ipynb

In [ ]:
%%writefile thermal_anomaly.py
from geopy.geocoders import Nominatim
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
import folium
from folium import Element
from folium.plugins import MarkerCluster
import urllib3
import warnings
from haversine_dist import distance
import os
from google.colab import userdata
warnings.filterwarnings("ignore")

NASA_FIRMS_KEY = os.environ["NASA_FIRMS_KEY"]
MAP_KEY = NASA_FIRMS_KEY

# Last two day VIIRS data for USA
#usa_url = 'https://firms.modaps.eosdis.nasa.gov/api/country/csv/' + MAP_KEY + '/MODIS_NRT/USA/7'
usa_url = 'https://firms.modaps.eosdis.nasa.gov/api/country/csv/' + MAP_KEY + '/VIIRS_SNPP_NRT/USA/2'
df_usa = pd.read_csv(usa_url)

# Get the shapefile to filter for states
# Read the shapefile
states_shapefile_url = 'https://firms.modaps.eosdis.nasa.gov/content/notebooks/exampleStatesShapefile2023(20m:coarse).zip'
states_gdf = gpd.read_file(states_shapefile_url)

#Convert df_usa data into a GeoDataFrame
df_usa_gdf = gpd.GeoDataFrame(
    df_usa,
    geometry=gpd.points_from_xy(df_usa['longitude'], df_usa['latitude']),
    crs=states_gdf.crs
    )

# Filter data to only include points within the shapefile, perform inner join on the usa_gdf and states_gdf
df_usa_within_shapefile = gpd.sjoin(df_usa_gdf, states_gdf, how="inner", predicate="intersects")

def get_thermal_anomaly(state,latitude_1, longitude_1):
  #Example: Filter data for user's state
  target_state = state
  subset_state = states_gdf[states_gdf['NAME'] == target_state]

  #Spatial join to get fire data witin user's state join usa fire data with state filtered shapefile as shapefile has the lat lon information
  df_usa_within_state = gpd.sjoin(df_usa_gdf, subset_state, how="inner", predicate="intersects")
  print(df_usa_within_state.head())
  count_nearby_wildfires = 0
  nearby_wildfires = []

  for row in df_usa_within_state.itertuples():
    # Get the latitude, longitude, and confidence of each fire within the state
    lat = row.latitude
    lon = row.longitude
    conf = row.confidence
    #Calculate the distance (using the havershine distance formula) of each fire from the user's location
    dist = distance(latitude_1, longitude_1, lat, lon)
    #Check if distance of the fire is less than or equal to 5 miles from the user's location, if yes, increase the count_nearby_wildfires by 1 and
    #add a dictionary of latitude,longitude, confidence, and distance of that wildfire to the nearby_wildfires list.
    if dist <= 20:
      count_nearby_wildfires += 1
      wildfire_info = {
          'latitude': lat,
          'longitude': lon,
          'confidence': conf,
          'distance': distance
      }
      nearby_wildfires.append(wildfire_info)

  return count_nearby_wildfires, nearby_wildfires

Writing thermal_anomaly.py


Calls the weather API with user's latitude and longtitude and get the high temperature, average precipation, average wind speed.

In [ ]:
%%writefile weather_data.py
import json
import re
import requests
from geopy.geocoders import Nominatim
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import urllib3
import warnings
warnings.filterwarnings('ignore')
from shapely import wkt

def get_weather_forecast(latitude, longitude):
    # Set a User-Agent header as required by the NWS API guidelines.
    headers = {"User-Agent": "my-weather-app (your_email@example.com)"}

    # Step 1: Get the metadata for the provided coordinates.
    # This call returns useful endpoints including the forecast URL.
    point_url = f"https://api.weather.gov/points/{latitude},{longitude}" #The first URL
    response = requests.get(point_url, headers=headers)

    # Check if the request was successful.
    if response.status_code != 200:
        raise Exception(f"Error fetching point data: {response.status_code} {response.text}")

    point_data = response.json()

    # Extract the forecast URL from the returned properties.
    forecast_url = point_data["properties"].get("forecast") #The second URL
    if not forecast_url:
        raise Exception("Forecast URL not found in the response.")

    # Step 2: Retrieve the forecast using the URL from the point data.
    forecast_response = requests.get(forecast_url, headers=headers)
    if forecast_response.status_code != 200:
        raise Exception(f"Error fetching forecast: {forecast_response.status_code} {forecast_response.text}")

    forecast_data = forecast_response.json()
    # Helper to find a period by number
    def get_period(periods, num):
        for p in periods:
            if p["number"] == num:
                return p
        raise ValueError(f"Period {num} not found")

    periods = forecast_data["properties"]["periods"]
    p1 = get_period(periods, 1)
    p2 = get_period(periods, 2)

    # High and low temperature between the two
    temps = [p1["temperature"], p2["temperature"]]
    high_temp = max(temps)
    low_temp = min(temps)

    # Average probability of precipitation (treat None as 0)
    pop_values = []
    for p in (p1, p2):
        v = p.get("probabilityOfPrecipitation", {}).get("value")
        pop_values.append(v if isinstance(v, (int, float)) else 0)
    avg_pop = sum(pop_values) / len(pop_values)

    # Parse windSpeed strings and average them
    def parse_wind_speed(ws: str) -> float:
        # Extract numbers
        nums = [float(n) for n in re.findall(r"\d+", ws)]
        return sum(nums) / len(nums)

    wind1 = parse_wind_speed(p1["windSpeed"])
    wind2 = parse_wind_speed(p2["windSpeed"])
    avg_wind = (wind1 + wind2) / 2

    #print(f"High temp: {high_temp}°F")
    #print(f"Low temp:  {low_temp}°F")
    #print(f"Avg precipitation chance: {avg_pop}%")
    #print(f"Avg wind speed: {avg_wind} mph")

    return high_temp, low_temp, avg_pop, avg_wind



Writing weather_data.py


In [ ]:
%%writefile year_month_season.py
from datetime import date
def get_year_month_season():
  current_date = date.today()

  year = current_date.year
  month = current_date.month
  season = None
  if month in [3, 4, 5]:
    #season = 'Spring'
    season = 1
  elif month in [6,7,8]:
    #season = 'Summer'
    season = 2
  elif month in [9,10,11]:
    #season = 'Fall'
    season = 3
  else:
    #season = 'Winter'
    season = 4
  return year, month, season

Writing year_month_season.py


Build Custom Machine Learning Model to train on the California Fire Dataset.

In [ ]:
%%writefile custom_model_weather_classification.py
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import statsmodels.api as sm
import scipy.stats as stats
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import precision_recall_fscore_support as score
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (confusion_matrix, accuracy_score)
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.utils import resample
#This dataset is fire dataset on California form 1948 to 2025 that has a mix of fire and non-fire rows
df = pd.read_csv('/content/sample_data/CA_Weather_Fire_Dataset_1984-2025_1.csv', low_memory=False)
df.head()

#Count of True and False in FIRE_START_DAY
df['FIRE_START_DAY'].value_counts()

df.shape

#Check for duplicates
df.duplicated().sum()

df.dtypes

#Convert the Date column which an object to a datetime
df["DATE"] = df["DATE"].apply(pd.to_datetime)

#Convert Season column which is an object to numerical values
df['SEASON'] = df['SEASON'].map({'Spring': 1, 'Summer': 2, 'Fall': 3, 'Winter': 4})

df.dtypes

# Set Date as the index
df.set_index('DATE', inplace=True)

#Check for columns with NA values
#df.isnull().sum().sort_values(ascending=False)

#drop the rows of the important columns with NA values
df = df.dropna(subset=['AVG_WIND_SPEED', 'WIND_TEMP_RATIO', 'PRECIPITATION', 'MIN_TEMP', 'MAX_TEMP', 'TEMP_RANGE'])
#df.isnull().sum().sort_values(ascending=False)


#Correlation heatmap
#plt.figure(figsize = (20, 12))
#sns.heatmap(df.corr(), cmap = "coolwarm", annot= True, vmin = 0, vmax = 1);

# drop columns that we will not use for this analysis
df = df.drop(columns = ['TEMP_RANGE', 'WIND_TEMP_RATIO','LAGGED_PRECIPITATION', 'LAGGED_AVG_WIND_SPEED', 'DAY_OF_YEAR'], axis=1)

df.describe()

##Drop YEAR == 2025
df = df[df['YEAR'] != 2025]

#Identify independent and dependent variables
X = df[['PRECIPITATION', 'MAX_TEMP', 'MIN_TEMP', 'AVG_WIND_SPEED', 'YEAR', 'MONTH', 'SEASON']]  # Feature selection
y = df['FIRE_START_DAY'] # dependent variable

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=33)

# Down‑sample the training set so that it has an equal number of “no‑fire” (0) and “fire” (1).
# Down‑sample only the training data—leaving the test set untouched so the evaluation remains honest.
# Re-combine X_train & y_train for down-sampling
train = X_train.copy()
train['FIRE_START_DAY'] = y_train.values

#Seperate majority/minority classes
df_majority = train[train['FIRE_START_DAY'] == 0]
df_minority = train[train['FIRE_START_DAY'] == 1]

#Down-sample majority (no-fire) to match minority (fire) count
df_majority_downsampled = resample(df_majority, replace=False, n_samples=len(df_minority), random_state=42)

#Re-combine and Shuffle
train_balanced = pd.concat([df_majority_downsampled, df_minority])
train_balanced = train_balanced.sample(frac=1).reset_index(drop=True)

#Split back into x_trained_balanced and y_trained_balanced
X_train_bal = train_balanced.drop('FIRE_START_DAY', axis=1)
y_train_bal = train_balanced['FIRE_START_DAY']


#Confirm balanced
print("Training class distribution after downsampling:")
print(y_train_bal.value_counts())

# Model Evaluation
# 1. Logistic Regression Classifier
# Fit logistic model on balanced training set
model = LogisticRegression(random_state=33, max_iter=1000).fit(X_train_bal, y_train_bal)


# Evaluate on untouched test set
y_pred = model.predict(X_test)

precision, recall, fscore, support = score(y_test, y_pred)

df_logit = pd.DataFrame({
    "labels": [0, 1],  # Adjusted labels for binary classification
    'precision': precision,
    "recall": recall,
    "fscore": fscore,
    "support": support
})

print(f"Logistic Regression: {df_logit}")

#Evaluate the model by looking at the Accuracy, Precision, Recall, and F1 score
print("n\Test set performance:")
print("Accuracy score of Logistic Regression:", accuracy_score(y_test, y_pred))
print("Precision score of Logistic Regression:", precision_score(y_test, y_pred))
print("Recall score of Logistic Regression :", recall_score(y_test, y_pred))
print("F1 score of Logistic Regression:", f1_score(y_test, y_pred))

#Plot the confusion matrix for Logisitic Regression Model
plt.figure(figsize=(6, 3))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title("Confusion Matrix - Logistic Regression")
plt.show()

# 2.Gradient Boosting Classifier
# Fit Gradient Boosting Classifier model on balanced training set
model_gb = GradientBoostingClassifier(random_state=33).fit(X_train_bal, y_train_bal)

#Evaluate on untouched test set
y_pred_gb = model_gb.predict(X_test)
print("n\Test set performance:")
print("Accuracy:", accuracy_score(y_test, y_pred_gb))


precision, recall, fscore, support = score(y_test, y_pred_gb)

df_gb = pd.DataFrame({
    "labels": [0,1],  # Adjusted labels for binary classification
    'precision': precision,
    "recall": recall,
    "fscore": fscore,
    "support": support
})

print(f"Gradient Boosting: {df_gb}")

#Evaluate the Gradient Boosting model by looking at the Accuracy, Precision, Recall, and F1 score
print("accuracy score of gradient boosting: ", accuracy_score(y_pred_gb, y_test))
print("precision score of gradient boosting: ", precision_score(y_pred_gb, y_test, average = "weighted"))
print("Recall score of gradient boosting: ", recall_score(y_pred_gb, y_test, average = "weighted"))
print("F1 score of gradient boosting: ", f1_score(y_pred_gb, y_test, average = "weighted"))

#Plot the confusion matrix for Gradient Boosting Model
plt.figure(figsize=(6, 3))
cm = confusion_matrix(y_test, y_pred_gb)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title("Confusion Matrix - Gradient Boosting")
plt.show()

#Code commented as fine tuning takes a long time to run
#Fine-tune Gradient Boosting Classifier
# model_gb = GradientBoostingClassifier(random_state=33)
# param_grid = {
#     "n_estimators":[80,100,120],
#     'max_depth': [3, 5, 10],          # Depth of each tree
#     'min_samples_split': [2, 5, 10],   # Minimum samples required to split a node
#     #"max_features": [6,7,8,9,10,11,12],
#     "min_samples_leaf": [1,2,3]
# }

# model_gb_grid1 = GridSearchCV(model_gb, param_grid, scoring = "f1_macro", cv = 10).fit(X_train_bal, y_train_bal)
# print("Best parameter are: %s"%model_gb_grid1.best_params_)
# print("Best F1-score:", model_gb_grid1.best_score_)

#Initialize the GradientBoostingClassifier by including the parameters from the fine-tuned model.
model_gb = GradientBoostingClassifier(random_state=0, max_depth=3, min_samples_leaf=1, min_samples_split=5, n_estimators=120).fit(X_train_bal, y_train_bal)
y_pred_gb = model_gb.predict(X_test)

precision, recall, fscore, support = score(y_test, y_pred_gb)

df_gb = pd.DataFrame({
    "labels": [0, 1],  # Adjusted labels for binary classification
    'precision': precision,
    "recall": recall,
    "fscore": fscore,
    "support": support
})

print(df_gb)

#Evaluate the model by looking at the Accuracy, Precision, Recall, and F1 score
print("accuracy score of gradient boosting: ", accuracy_score(y_pred_gb, y_test))
print("precision score of gradient boosting: ", precision_score(y_pred_gb, y_test, average = "weighted"))
print("Recall score of gradient boosting: ", recall_score(y_pred_gb, y_test, average = "weighted"))
print("F1 score of gradient boosting: ", f1_score(y_pred_gb, y_test, average = "weighted"))

#Plot the confusion matrix for Fine-tuned Gradient Boosting Model
plt.figure(figsize=(6, 3))
cm = confusion_matrix(y_test, y_pred_gb)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title("Confusion Matrix - Fine-tuned Gradient Boosting Model")
plt.show()

#Function to get prediction
def get_prediction(precipitation, max_temp, min_temp, avg_wind_speed, year, month, season):
  X_loc = pd.DataFrame({'PRECIPITATION': [precipitation], 'MAX_TEMP':[max_temp], 'MIN_TEMP':[min_temp], 'AVG_WIND_SPEED':[avg_wind_speed], 'YEAR':[year], 'MONTH':[month], 'SEASON':[season]})
  prediction = model_gb.predict(X_loc)

  if prediction == 0:
    prediction = "Very Low Risk"
  else:
    #Get the prediction probabilities if the the prediction is 1
    probs = model_gb.predict_proba(X_loc)
    if probs[1] < 0.25:
      prediction = "Low Risk"
    elif probs[1] >= 0.25 and probs[1] < 0.5:
      prediction = "Moderate Risk"
    elif probs[1] >= 0.5 and probs[1] < 0.85:
      prediction = "High Risk"
    else:
      prediction = "Extreme Risk"

  return prediction

Writing custom_model_weather_classification.py


In [ ]:
%%writefile app.py
import streamlit as st
from google.colab import userdata
import os
import sys
sys.path.append('/content/')
from user_intent import user_intent
from RAG_LLM import qa_conversation
from get_coordinates import get_coordinates
from city_country_ner import ner
from get_state import get_state
from weather_api_alerts import weather_api_alerts
from state_abbreviation import get_state_abbr
from thermal_anomaly import get_thermal_anomaly
from weather_data import get_weather_forecast
from year_month_season import get_year_month_season
from custom_model_weather_classification import get_prediction
import pandas as pd
import json
import requests
from transformers import pipeline

# Streamlit frontend
# Set the title of the app
st.title("Wildfire AI Assistant")

# Initialize chat history
if "messages" not in st.session_state:
    st.session_state.messages = []

# Display chat messages from history
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# User input
#Show the chat input and capture the user’s response
prompt = st.chat_input("Hello, I am your Wildfire AI Assistant, how can I help you today?")

#Only proceed if the user actually entered text
if prompt:
    # Here add the user message to the chat history, generate a response, display it, etc.
    # Add user message to chat history
    st.session_state.messages.append({"role": "user", "content": prompt})
    # Display user message in chat message container
    with st.chat_message("user"):
        st.markdown(prompt)

    #Use NLP zero-shot classification "facebook/bart-large-mnli" to check what the user is asking for by calling the user_intent function and passing the user's prompt to it.
    user_intent = user_intent(prompt)
    answer = ""

    #If the user ask for a predicition of wildifire in a specific area, then do 3 things:
    #1. Call the weather api alerts endpoint to get the active fire alerts.
    #2. Check NASA FIRMS Thermal Anomaly.
    #3. Call the custom classification model and get the prediction.
    if user_intent == "area-specific wildfire risk prediction":

      #1. Call the weather api alerts endpoint to get the active fire alerts
      #Figure out the place name from the user's prompt by calling the ner function and pass the user's prompt
      place_name = ner(prompt)
      #If there is no place name, call answer = qa_conversation(prompt) and dont execute the rest of this code
      if place_name:

          #Figure out the user's location - longitude, latitude, and address from a place name eg. Santa Fe, NM by calling the function get_coordinates
          coordinates = get_coordinates(place_name)

          address = ""
          if coordinates:
            latitude, longitude, address = coordinates

          #Get the state name using a zero shot classifier "facebook/bart-large-mnli" from address by calling the get_state function
          state = get_state(address)

          #Get the state abbreviation using the get_state_abbr function and pass the state
          state_abbreviation = get_state_abbr(state)

          #Build the URL with state_abbreviation to call the weather.gov api for alerts
          weather_alerts_api = "https://api.weather.gov/alerts/active?area=" + state_abbreviation

          #Call the endpoint and process the data
          response = requests.get(weather_alerts_api)
          #Convert the response to a json
          data = response.json()

          #Call the weather_api_alerts to extract the Red Flag Warning and Fire Weather Watch events
          alerts = weather_api_alerts(data)

          alert_messages = ""
          #Check if the return is a string or a list
          #Build the alert message to return to the user.
          if isinstance(alerts, str):
            alert_messages = alerts
          else:
            alert_messages += "Here are the wildfire alerts for your state today -  "  + "\n"
            for alert in alerts:
              #alert_messages += "Here are the wildfire alerts for your state today -  "  + "\n"
              alert_messages += "Area Description: " + str(alert['areaDesc']) + "\n" + "\n"
              alert_messages += "Headline: " + str(alert['headline'] ) + "\n" + "\n"
              alert_messages += "Description: " + str(alert['description'] ) + "\n" + "\n"
              # Check if alert['instruction'] is not null
              if alert['instruction'] is not None:
                alert_messages += "Instruction: " + str(alert['instruction'] ) + "\n" + "\n"

          #2. Check NASA FIRMS Thermal Anomaly by calling the get_thermal_anomaly function and pass the state, latitude, and longitude.
          # This will return a count of nearby wildifres and a list of dictionaries containing nearby wildfire information
          count_nearby_wildfires, nearby_wildfires = get_thermal_anomaly(state,latitude, longitude)

          #Check if nearby_wildfire is an empty list or there is information in it
          nasa_firms_thermal_anomaly = ""
          #if count_nearby_wildfires > 0:
          if nearby_wildfires:
            clean = []
            #nearby_wildfires is a list of dicts - here we get it in a nice table and and add it to the streamlit frontend.
            for rec in nearby_wildfires:
            # drop any key whose value is callable
              clean.append({k:v for k,v in rec.items() if not callable(v)})

            details_str = json.dumps(clean, indent=2)
            df = pd.DataFrame(clean)
            #add an index column starting at 1
            df.index = df.index + 1
            df.index.name = "#"

            # show count + table of nasa firms thermal anomaly - near real time fire information,
            with st.chat_message("assistant"):
              st.write(f"There are {count_nearby_wildfires} wildfires within 20 miles of your location.")
              st.table(df)   # formatted table

          else:
            #nasa_firms_thermal_anomaly = "There are no wildfires within 5 miles of your location."
            with st.chat_message("assistant"):
                    st.write("There are no wildfires within 20 miles of your location.")

          #3. Call the custom classification model and get the prediction
          #Get weather data (calls weather API) based on latitude and longitude of the user's location call get_weather_forecast function
          high_temp, low_temp, avg_pop, avg_wind = get_weather_forecast(latitude, longitude)

          #Get the current year, month, and season by calling the get_year_month_season function
          year, month, season = get_year_month_season()

          #Get the classification from the custom model by calling the get_prediction function
          prediction = get_prediction(avg_pop, high_temp, low_temp, avg_wind, year, month, season)

          #Build answer with alert_messages from api.weather.gov/alerts, and prediction from the custom classification model.
          # append prediction text
          answer = f"{alert_messages}\nYour area's wildfire risk for today is **{prediction}**."

      #If a place name has not been found call rag chain
      else:
        answer = qa_conversation(prompt)

    #Else if the user is asking for general wildfire information
    # as determined by zero shot classification, including risk awareness, prevention, and preparedness guidance, then call the RAG chain.
    else:

      #Pass the prompt to the conversational retrival chain and get the chatbot response
      answer = qa_conversation(prompt)

    # Add chatbot response to chat history
    st.session_state.messages.append({"role": "assistant", "content": answer})
    #st.session_state.messages.append({"role": "assistant", "content": user_intent})
    # Display chatbot response in chat message container
    with st.chat_message("assistant"):
        st.markdown(answer)


Writing app.py


In [ ]:
#!npm install localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋
added 22 packages in 2s
⠋
⠋3 packages are looking for funding
⠋  run `npm fund` for details
⠋

In [ ]:
#!streamlit run app.py &>/content/logs.txt & npx localtunnel --port 8501 & curl ipv4.icanhazip.com

34.143.137.70
⠙your url is: https://hungry-chicken-allow.loca.lt


In [ ]:
from pyngrok import ngrok
import time
import threading
import os
from pyngrok import ngrok
from google.colab import userdata

NGROK_AUTH_TOKEN = userdata.get("ngrok_auth")
os.environ["NGROK_AUTH_TOKEN"] = NGROK_AUTH_TOKEN


# Add authtoken
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Kill any previous tunnels
ngrok.kill()

# Start Streamlit in the background
def run_streamlit():
    os.system("streamlit run combined_copilot.py --server.headless true --server.port 8501")

thread = threading.Thread(target=run_streamlit)
thread.start()

time.sleep(5)  # wait for Streamlit to boot

# Open public URL via ngrok
public_url = ngrok.connect(8501)
print("Streamlit app running at:", public_url)